# Qwen4B — vector truy vấn runtime trên Kaggle

Notebook mới, tách khỏi thí nghiệm BGE/Qwen cũ. Tạo **30 vector** cho mục tiêu kiến thức và câu hỏi chẩn đoán đã biên soạn của 10 concept pilot. Không huấn luyện model, không đánh giá lại Recall, không mở dịch vụ/tunnel. Không tải dữ liệu học sinh, lời giải chuẩn, `.env` hoặc toàn bộ repo.

1. Import notebook này vào Kaggle; chọn **GPU T4 ×2**, bật **Internet**.
2. Add Input → upload **`pilot-query-vectors-qwen3-4b-input.zip`** từ `data/exports/`. Kaggle có thể tự giải nén ZIP; cả hai dạng đều được hỗ trợ.
3. Bắt đầu **session mới**, Save Version → Save & Run All. Không chạy chung model BGE/Qwen của notebook khác để tránh hết VRAM.
4. Tải **`pilot-query-vectors-qwen3-4b.zip`** trong Output. Đặt vào `data/exports/`, chạy lệnh import bên dưới.

Model pin revision và FP16/last-token/L2 như lần chạy Qwen thành công. Lần đầu Kaggle vẫn cần tải khoảng 8 GB trọng số; không tải chúng về máy cá nhân. Batch nhỏ nên phần encode ngắn, phần tải/nạp model có thể chiếm phần lớn thời gian.

Backend chỉ lấy vector khi query khớp chính xác danh mục và phiên bản nội dung. Query mới ngoài danh mục dùng lexical fallback có nhãn; đây không phải encode online mọi câu hỏi tự do.


In [1]:
# Validate local-generated manifest and every file BEFORE executing bundled code.
from pathlib import Path
import hashlib, json, sys, uuid, zipfile
PREFIX = 'pilot-query-vectors-qwen3-4b/'
INPUT_FILES = ['backend/app/__init__.py', 'backend/app/contracts.py', 'backend/app/curriculum.py', 'backend/app/pilot_contracts.py', 'backend/app/retrieval_contracts.py', 'backend/app/retrieval.py', 'backend/app/local_embedding.py', 'research/retrieval/evaluate.py', 'research/retrieval/requirements.txt', 'research/retrieval/qwen3_4b/experiment.py', 'research/retrieval/qwen3_4b/requirements.txt', 'backend/app/qwen_query.py', 'research/retrieval/qwen3_4b/runtime_catalog.py', 'specification.json', 'manifest.json']
EXPECTED_MANIFEST_SHA256 = '6ff0cc1602b8b69cb42b7b390a552354caf8c2e1ee82c02123e58ebcdbbf184d'
candidates = list(Path('/kaggle/input').rglob('pilot-query-vectors-qwen3-4b-input.zip'))
if not candidates:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('manifest.json')
                  if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256]
if len(candidates) != 1:
    raise ValueError('Add exactly one matching runtime input ZIP/dataset; remove old or duplicate input versions.')
SOURCE = candidates[0]
limit = 5 * 1024**2
if SOURCE.is_file():
    if SOURCE.stat().st_size > limit:
        raise ValueError('Input ZIP too large')
    with zipfile.ZipFile(SOURCE) as z:
        expected = {PREFIX + n for n in INPUT_FILES}
        if set(z.namelist()) != expected or len(z.namelist()) != len(expected) or sum(i.file_size for i in z.infolist()) > limit:
            raise ValueError('Input allowlist/size mismatch')
        raw = {n: z.read(PREFIX + n) for n in INPUT_FILES}
else:
    if {p.relative_to(SOURCE).as_posix() for p in SOURCE.rglob('*') if p.is_file()} != set(INPUT_FILES):
        raise ValueError('Input directory allowlist mismatch')
    if sum((SOURCE/n).stat().st_size for n in INPUT_FILES) > limit:
        raise ValueError('Input contents too large')
    raw = {n: (SOURCE/n).read_bytes() for n in INPUT_FILES}
if hashlib.sha256(raw['manifest.json']).hexdigest() != EXPECTED_MANIFEST_SHA256:
    raise ValueError('Notebook/input mismatch: upload the pair generated together.')
manifest = json.loads(raw['manifest.json'])
if manifest['sha256'] != {n: hashlib.sha256(v).hexdigest() for n,v in raw.items() if n != 'manifest.json'}:
    raise ValueError('Input hash mismatch')
WORK = Path('/kaggle/working') / ('qwen-runtime-' + uuid.uuid4().hex)
for n,v in raw.items():
    target = WORK/n
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(v)
sys.path[:0] = [str(WORK/'backend'), str(WORK)]
print('Validated input; queries:', len(json.loads(raw['specification.json'])['queries']))


Validated input; queries: 30


In [2]:
# Keep Kaggle's preinstalled CUDA PyTorch. Do not install the CPU torch build.
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(WORK/'research/retrieval/qwen3_4b/requirements.txt')])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


0

In [3]:
from research.retrieval.qwen3_4b.runtime_catalog import run
from research.retrieval.qwen3_4b.experiment import gpu_memory
print(gpu_memory())
OUTPUT = Path('/kaggle/working') / 'pilot-query-vectors-qwen3-4b.zip'
result = run(SOURCE, OUTPUT, Path('/kaggle/working/qwen3-4b-model-cache'))
print(json.dumps(result, ensure_ascii=False, indent=2))


[{'device': 0, 'name': 'Tesla T4', 'free_bytes': 15526068224, 'total_bytes': 15636037632}, {'device': 1, 'name': 'Tesla T4', 'free_bytes': 15526068224, 'total_bytes': 15636037632}]


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

... (2 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

{
  "output": "/kaggle/working/pilot-query-vectors-qwen3-4b.zip",
  "queries": 30,
  "dimensions": 2560,
  "timings": {
    "model_load_seconds": 52.299,
    "query_encode_seconds": 3.025
  },
  "peak_gpu_memory_mib": {
    "0": 3840.9,
    "1": 3869.6
  }
}


In [4]:
from IPython.display import FileLink, display
display(FileLink(str(OUTPUT)))


/kaggle/working/pilot-query-vectors-qwen3-4b.zip

## Sau khi tải về máy

Đặt ZIP trong `data/exports/`. Từ thư mục gốc dự án chạy:

```powershell
.\.venv\Scripts\python.exe research/retrieval/qwen3_4b/runtime_catalog.py import --input data/exports/pilot-query-vectors-qwen3-4b.zip
```

Importer chỉ đọc JSON, kiểm tra hash/model/revision/danh mục/phiên bản hiện tại trước khi lưu `data/retrieval/qwen3-4b-runtime/catalog.json`. Không ghi đè catalog khác. Không nạp code tải về và không sửa corpus/index/kết quả nghiên cứu cũ. Backend dùng `QWEN_QUERY_PROVIDER=catalog`; pgvector chứa 40 vector nội dung cần được nạp riêng bằng `rag-init`.
